# Exploration — student-health

Notebook-first iteration on this project, using the same MLflow tracking the CLI uses.

| Step | What | API |
|------|------|-----|
| 1 | Peek at processed features | `DataStore.preview()` |
| 2 | Try a quick idea inline | `kitchen.experiment(exploratory=True)` |
| 3 | Run a Trainer with tracking | `kitchen.init_run(exploratory=True, log_model=False)` |
| 4 | Compare runs | `kitchen leaderboard` |

**Prerequisite:** run `kitchen run features` first so `data/processed/` exists.

Exploratory runs are tagged `run_type=exploratory`, so they stay separable from
pipeline runs: `kitchen leaderboard --exclude-exploratory` hides them and
`--only-exploratory` shows just these notebook sketches.

In [ ]:
import yaml
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import kitchen
from kitchen.store import DataStore

with open("menu.yaml") as f:
    params = yaml.safe_load(f)

EXPERIMENT = params.get("experiment", "student-health")
TARGET = params.get("model", {}).get("target", "target")
PROCESSED_FILE = params.get("features", {}).get("processed_file", "features.parquet")

store = DataStore()
print(f"experiment={EXPERIMENT}  target={TARGET}  features={PROCESSED_FILE}")

## Step 1 — Peek at the processed features

`preview()` searches `processed/` first, then `raw/`, and returns the first rows —
just the filename, no path juggling.

In [ ]:
store.preview(PROCESSED_FILE)

## Step 2 — Try a quick idea with `kitchen.experiment()`

Write model code directly in the cell — no `Trainer` subclass needed. `exploratory=True`
tags the run so it stays out of the default leaderboard ranking.

**Metric naming matters:** `kitchen leaderboard` ranks by the metric in your
`thresholds:` section. Log under that exact name, or the run won't appear in the
default view (`kitchen leaderboard --metric <name>` still finds it).

In [ ]:
df = store.load_parquet(PROCESSED_FILE)
if TARGET not in df.columns:
    raise ValueError(
        f"target column {TARGET!r} is not in the features ({list(df.columns)}). "
        "Set model.target in menu.yaml (or change TARGET in the setup cell)."
    )

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

with kitchen.experiment(EXPERIMENT, run_name="nb-logreg", exploratory=True) as run:
    model = LogisticRegression(max_iter=200)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_val, model.predict(X_val))
    run.log(val_accuracy=acc)

print(f"val_accuracy: {acc:.4f}  (run {run.run_id})")

## Step 3 — Run a Trainer with `kitchen.init_run()`

`init_run()` injects the same MLflow context that `kitchen run train` uses, so you can
iterate on a `Trainer` in the notebook. `log_model=False` keeps these throwaway runs out
of the model registry; `exploratory=True` tags them.

The `SimpleTrainer` below is a stand-in so this cell runs on a fresh project. In your
project, swap it for your own: `from src.train.run import StudentHealthTrainer`.

In [ ]:
import mlflow

from kitchen.steps import Trainer


class SimpleTrainer(Trainer):
    """Stand-in Trainer — replace with StudentHealthTrainer from src/train/run.py."""

    def fit(self, df, params):
        X = df.drop(columns=[TARGET])
        y = df[TARGET]
        X_tr, X_vl, y_tr, y_vl = train_test_split(X, y, test_size=0.2, random_state=42)
        model = LogisticRegression(max_iter=300)
        model.fit(X_tr, y_tr)
        mlflow.log_metric("val_accuracy", accuracy_score(y_vl, model.predict(X_vl)))
        return model


with kitchen.init_run(params, run_name="nb-trainer", exploratory=True, log_model=False) as tracker:
    SimpleTrainer().run(store, tracker, params)

print("Logged — see it with: kitchen leaderboard --only-exploratory")

## Step 4 — Compare runs

From a terminal in this project directory:

```bash
kitchen leaderboard                       # ranked runs
kitchen leaderboard --only-exploratory    # just these notebook sketches
kitchen leaderboard --exclude-exploratory # just pipeline runs
kitchen diff <run_a> <run_b>              # what changed between two runs
kitchen ui                                # open the MLflow UI
```

Found a keeper? Promote it: `kitchen promote --run-id <run_id>`.